# Model Training for Parkinson’s Disease Detection

This notebook trains a **Gradient Boosting Classifier** for Parkinson’s Disease detection using voice-based biomedical features.

The workflow includes:
- loading and cleaning the dataset
- splitting and scaling the data
- balancing the training set with SMOTETomek
- hyperparameter tuning with GridSearchCV
- evaluating the final model on an unseen test set
- saving the trained model and scaler

In [1]:
from src.data_utils import (
    load_data,
    clean_numeric_data,
    split_and_scale,
    balance_classes
)

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    matthews_corrcoef,
    confusion_matrix
)

from xgboost import XGBClassifier 

import os
import joblib

## 1. Data Loading and Preprocessing

In this step, the dataset is loaded and cleaned by keeping only numeric features.  
The data is then split into training and test sets using stratified sampling to preserve class distribution.  
Feature scaling is applied using `StandardScaler`, and class imbalance is handled with **SMOTETomek** on the training set only.

In [2]:
df = load_data(r"D:\Datasets\PD-alldata.csv")
df = clean_numeric_data(df, target_column="status")

X_train, X_test, y_train, y_test, scaler = split_and_scale(
    df,
    target_column="status"
)

X_train_balanced, y_train_balanced = balance_classes(
    X_train,
    y_train,
    method="tomek"
)

feature_names = X_train.columns.tolist()

print("Train shape:", X_train_balanced.shape)
print("Test shape:", X_test.shape)
print("Balanced training labels:")
print(y_train_balanced.value_counts())
print("Number of features:", len(feature_names))

Train shape: (296, 19)
Test shape: (46, 19)
Balanced training labels:
status
1    148
0    148
Name: count, dtype: int64
Number of features: 19


## 2. Hyperparameter Tuning with Cross-Validation

To improve model robustness, `GridSearchCV` is used to tune the Gradient Boosting model.  
The search is performed using **5-fold cross-validation** and optimized with **ROC-AUC**, which is suitable for imbalanced classification tasks.

In [3]:
param_grid = {
    "n_estimators": [50, 100, 150],
    "learning_rate": [0.05, 0.1, 0.2],
    "max_depth": [2, 3, 4]
}

In [4]:
grid_search = GridSearchCV(
    estimator=GradientBoostingClassifier(random_state=42),
    param_grid=param_grid,
    scoring="roc_auc",
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_balanced, y_train_balanced)

best_model = grid_search.best_estimator_

print("Best Parameters:")
print(grid_search.best_params_)

print("\nBest Cross-Validation ROC-AUC:")
print(grid_search.best_score_)

print("\nBest model:")
print(best_model)

Fitting 5 folds for each of 27 candidates, totalling 135 fits
Best Parameters:
{'learning_rate': 0.1, 'max_depth': 4, 'n_estimators': 100}

Best Cross-Validation ROC-AUC:
0.9885517241379311

Best model:
GradientBoostingClassifier(max_depth=4, random_state=42)


## 3. Test Set Prediction

After selecting the best hyperparameters through cross-validation, the optimized model is used to generate predictions on the unseen test set.

In [5]:
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

## 4. Model Evaluation

The model is evaluated using several performance metrics:
- Accuracy
- Precision
- Recall
- F1-score
- ROC-AUC
- Matthews Correlation Coefficient (MCC)

The confusion matrix is also reported to show detailed classification performance.

In [6]:
results = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1-score": f1_score(y_test, y_pred),
    "ROC-AUC": roc_auc_score(y_test, y_proba),
    "MCC": matthews_corrcoef(y_test, y_pred)
}

print("Test Results:")
for metric, value in results.items():
    print(f"{metric}: {value:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Test Results:
Accuracy: 0.9565
Precision: 0.9737
Recall: 0.9737
F1-score: 0.9737
ROC-AUC: 0.9868
MCC: 0.8487

Confusion Matrix:
[[ 7  1]
 [ 1 37]]


## 5. Result Interpretation

The model achieves strong classification performance on the test set, with high ROC-AUC, precision, and recall.  
Using cross-validation for model selection improves the reliability of the final results and reduces the risk of overfitting.

Because the dataset is relatively small and imbalanced, metrics such as **ROC-AUC** and **MCC** are especially important for a more reliable assessment.

## 6. Save Trained Artifacts

Finally, the trained model and scaler are saved so they can be reused later for inference, visualization, or deployment.

In [7]:
os.makedirs("models", exist_ok=True)

joblib.dump(best_model, "models/gb_pd_alldata_v1.pkl")
joblib.dump(scaler, "models/scaler_pd_alldata_v1.pkl")

print("Model and scaler saved successfully")

Model and scaler saved successfully


## 7. XGBoost Model Training

In addition to Gradient Boosting, XGBoost is evaluated as an alternative ensemble method.  
XGBoost is known for its efficiency and strong performance on structured/tabular data.

In [8]:
xgb_model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=4,
    random_state=42,
    eval_metric="logloss"
)

xgb_model.fit(X_train_balanced, y_train_balanced)

y_pred_xgb = xgb_model.predict(X_test)
y_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]

## 8. XGBoost Evaluation

The XGBoost model is evaluated using the same metrics for fair comparison.

In [9]:
print("=== XGBoost Results ===")
print("Accuracy :", accuracy_score(y_test, y_pred_xgb))
print("Precision:", precision_score(y_test, y_pred_xgb))
print("Recall   :", recall_score(y_test, y_pred_xgb))
print("F1-score :", f1_score(y_test, y_pred_xgb))
print("ROC-AUC  :", roc_auc_score(y_test, y_proba_xgb))
print("MCC      :", matthews_corrcoef(y_test, y_pred_xgb))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_xgb))

=== XGBoost Results ===
Accuracy : 0.9782608695652174
Precision: 0.9743589743589743
Recall   : 1.0
F1-score : 0.987012987012987
ROC-AUC  : 0.9703947368421053
MCC      : 0.9233439784631201

Confusion Matrix:
[[ 7  1]
 [ 0 38]]


## 9. Model Comparison

This section compares Gradient Boosting and XGBoost performance on the test set.

In [10]:
comparison = {
    "Model": ["Gradient Boosting", "XGBoost"],
    "Accuracy": [
        accuracy_score(y_test, y_pred),
        accuracy_score(y_test, y_pred_xgb)
    ],
    "ROC-AUC": [
        roc_auc_score(y_test, y_proba),
        roc_auc_score(y_test, y_proba_xgb)
    ],
    "MCC": [
        matthews_corrcoef(y_test, y_pred),
        matthews_corrcoef(y_test, y_pred_xgb)
    ]
}

import pandas as pd
comparison_df = pd.DataFrame(comparison)
comparison_df

,Model,Accuracy,ROC-AUC,MCC
0,Gradient Boosting,0.956522,0.986842,0.848684
1,XGBoost,0.978261,0.970395,0.923344


## 10. Final Model Comparison Interpretation

XGBoost achieved higher accuracy and MCC, indicating stronger classification performance and better handling of class imbalance.  

Gradient Boosting achieved slightly higher ROC-AUC, suggesting better probability separation between classes.

This demonstrates a trade-off between classification accuracy and probabilistic discrimination, and highlights the importance of evaluating multiple metrics when working with imbalanced datasets.